In [ ]:
from aereo.builtins.search import search_earthaccess
from aereo.builtins.task_builder import build_grouped_tasks
from aereo.cache import TaskResultCache
from aereo.executors import LocalExecutor
from aereo.pipeline import ExtractionJob

job = ExtractionJob.load_from_config(
    config_dir="config",
    config_name="job_sentinel3",
)

In [ ]:
from datetime import datetime, timezone

assets = search_earthaccess(
    collections={"S3A_OL_1_EFR": ["Oa08", "Oa17"]},
    intersects="config/aoi/chocon.geojson",
    start_datetime=datetime(2024, 1, 1, tzinfo=timezone.utc),
    end_datetime=datetime(2024, 1, 1, 23, 59, 59, tzinfo=timezone.utc),
)
print(f"✓ Found {len(assets)} scenes")

In [ ]:
tasks = build_grouped_tasks(
    search_results=assets,
    job=job,
    cells_per_task=5,
)
print(f"✓ Built {len(tasks)} tasks")

In [ ]:
artifacts = job.execute(
    tasks,
    executor=LocalExecutor(workers=1, cache=TaskResultCache()),
)
print(f"✓ Extracted {len(artifacts)} artifacts")

In [ ]:
from aereo.viz import plot_artifact_patches

plot_artifact_patches(artifacts, ds_factor=1)